# Voice AI TTS Project
This notebook trains a custom voice model using Coqui TTS with English, Hindi, and Hinglish audio.

## Step 1: Install Dependencies

In [ ]:
!pip install TTS==0.10.2 torch==1.10.0 numpy pandas pydub ffmpeg-python --quiet

## Step 2: Upload and Organize Files

In [ ]:

import os
import shutil
import pandas as pd

# Create necessary folders
os.makedirs("clips", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Move all .wav files into /clips if needed
for f in os.listdir("/content"):
    if f.endswith(".wav"):
        shutil.move(f, f"clips/{f}")


## Step 3: Fix and Load Metadata

In [ ]:

# Fix pipe delimiter if needed
with open("metadata.csv", "r") as f:
    lines = [line.strip().replace("|", ",") for line in f if line.strip()]

with open("metadata.csv", "w") as f:
    for line in lines:
        f.write(line + "\n")

# Load and check metadata
df = pd.read_csv("metadata.csv")
df.head()


## Step 4: Configure Model and Train

In [ ]:

from TTS.tts.configs.tacotron2_config import Tacotron2Config
from TTS.tts.models.tacotron2 import Tacotron2
from TTS.config.shared_configs import BaseDatasetConfig, BaseAudioConfig
from TTS.trainers import Trainer

# Dataset config
dataset_config = BaseDatasetConfig(
    meta_file_train="metadata.csv",
    path="clips"
)

# Audio config
audio_config = BaseAudioConfig(
    sample_rate=22050,
    fft_size=1024,
    num_mels=80
)

# Tacotron2 config
config = Tacotron2Config(
    model_name="voiceai_model",
    run_name="run_01",
    output_path="outputs",
    batch_size=4,
    eval_batch_size=4,
    num_loader_workers=0,
    run_eval=True,
    test_delay_epochs=2,
    save_checkpoints=True,
    save_best_after=2,
    mixed_precision=False,
    output_per_step=5,
    print_step=25,
    datasets=[dataset_config],
    audio=audio_config
)

# Train model
model = Tacotron2.init_from_config(config)
trainer = Trainer(config, model=model)
trainer.fit()


## Step 5: Generate Sample Voice

In [ ]:

from TTS.utils.synthesizer import Synthesizer

# Load trained model
synthesizer = Synthesizer(
    tts_checkpoint="outputs/run_01/checkpoint.pth",
    tts_config_path="outputs/run_01/config.json"
)

# Generate sample
output_wav = synthesizer.tts("Hello Vikranth! This is your custom voice model.")
synthesizer.save_wav(output_wav, "sample_output.wav")
